# Next step: step 8a follow-up, the seed in `SimulatorConfig`
**What it changes:** the seed moves into `SimulatorConfig` (`seed`, 0 by default), so a config holds everything that
sets a simulation's result. `sim`'s functions lose their `seed=` argument, and `simr` uses each simulator entry's seed
for every model. `run.seed`, `simr.model_seed_for` and `SimrResult.seed` are gone.

| Before | Now |
|---|---|
| `sim.simulate_particles(params, config, seed=1)` | `sim.simulate_particles(params, replace(config, seed=1))` |
| `seed=None` (the default): random every run | `seed=0` (the default): the same particles every run |
| `simr`: `run.seed`, and a different seed per model from `SeedSequence([run seed, m])` | `simr`: each entry's `seed`, the same for every model |
| Compare draws: change `run.seed` | Compare draws: entries that differ only in `seed` |

**Why:** SPAM draws the same random numbers for every model (each particle's ring, radius, angle and heat kicks), so
with one seed every model's disks share one pattern, stretched by `rout` and tilted by `theta` and `phi`. Score
differences between models then come only from their parameters. A different seed is only needed to test or show how
much the draw matters, and those simulations can sit side by side in one model's folder (Step 8b), each with its seed.

## What changed
**`src/galactic_tidal_constraints/simr/sim.py`:** the new field, checked like the others, and passed to `jspam` with the
rest of the config:
```python
@dataclass(frozen=True)
class SimulatorConfig:
    seed: int = 0
    n1: int = 1000
    ...

def _jspam_args(params, config, t_start=None):
    return {
        "t_start": ...,
        "seed": config.seed,
        "n1": config.n1,
        ...
    }

def simulate_particles(params, config=SimulatorConfig()):   # no seed argument
    return jspam.run_collision(_spam_params(params), **_jspam_args(params, config))
```
`initial_state` and `unperturbed_particles` change the same way.

**`src/galactic_tidal_constraints/simr/simr.py`:** no run seed; each entry's config already holds its seed:
```python
for m, params in enumerate(models):
    for s, config in enumerate(configs):
        for name in wanted:
            data[name][m, s] = SIM_DATA[name](params, config)
return SimrResult(models=models, simulators=configs, data=data)
```

**Tests:** `test_sim.py` (Step 3: the default is 0 and repeats; one seed gives every model the same disk pattern),
`test_simr.py`. **Docs:** `docs/spam_physics.md`, `seed`. **Decision:** `PLAN.md`, Models and stage lists.

In [ ]:
%matplotlib inline
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np

from galactic_tidal_constraints.simr import sim, simr

# Line 3 of JSPAM's archive/data.txt
params = np.array([1.08119, -1.66798, -2.06826, -0.18438, -0.39693, -0.02919, 0.51654, 0.36192, 0.94043, 0.43877,
                   12.97415, 135.11902, 97.54035, 305.01225])
config = sim.SimulatorConfig(n1=1500, n2=1500)
N = config.n1

## 1. The default seed is 0, and runs repeat

In [ ]:
print("default seed:", sim.SimulatorConfig().seed)
first = sim.simulate_particles(params, config)
again = sim.simulate_particles(params, config)
other = sim.simulate_particles(params, replace(config, seed=1))
print("same config, identical particles:", np.array_equal(first, again))
print("seed 1, different particles:     ", not np.array_equal(first[:-1], other[:-1]))
print("seed 1, same secondary's center: ", np.array_equal(first[-1], other[-1]))

## 2. One seed, one pattern for every model
The primary disk of the model, laid flat (`theta1` = `phi1` = 0), next to the same model with `rout1` doubled. Each
particle keeps its angle and its fractional radius, so the pattern is the same, only stretched. Colors follow particle
order, so matching colors are the same particle:

In [ ]:
flat = params.copy()
flat[[sim.PARAM_INDEX["theta1"], sim.PARAM_INDEX["phi1"]]] = 0.0
wider = flat.copy()
wider[sim.PARAM_INDEX["rout1"]] *= 2

disks = {"rout1 as given": sim.unperturbed_particles(flat, config)[:N],
         "rout1 doubled": sim.unperturbed_particles(wider, config)[:N]}
fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
for ax, (title, disk) in zip(axes, disks.items()):
    ax.scatter(disk[:, 0], disk[:, 1], c=np.arange(N), cmap="viridis", s=2)
    ax.set_title(title)
    ax.set_aspect("equal")
plt.show()

a, b = disks.values()
rin, rout = 0.05, flat[sim.PARAM_INDEX["rout1"]]
print("angles identical:            ", np.allclose(np.arctan2(a[:, 1], a[:, 0]), np.arctan2(b[:, 1], b[:, 0])))
print("fractional radii identical:  ", np.allclose((np.hypot(a[:, 0], a[:, 1]) - rin) / (rout - rin),
                                                   (np.hypot(b[:, 0], b[:, 1]) - rin) / (2 * rout - rin)))

## 3. Seeds in `simr`
Every model uses its entry's seed. To compare draws, list entries that differ only in `seed`:

In [ ]:
heavier = params.copy()
heavier[sim.PARAM_INDEX["mass2"]] *= 2
request = {"run": {"return": ["sim.unperturbed"]}, "simulator": [{"n1": N, "n2": N, "seed": s} for s in (0, 1, 2)]}
result = simr.run(np.stack([params, heavier]), request)
print("seeds:", [c.seed for c in result.simulators])
starts = result.data["sim.unperturbed"]
print("models 0 and 1 (only mass2 differs), primary disks identical:", np.array_equal(starts[0, 0][:N], starts[1, 0][:N]))

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)
for ax, config_s, start in zip(axes, result.simulators, starts[0]):
    ax.scatter(start[:N, 0], start[:N, 1], s=1)
    ax.set_title(f"seed {config_s.seed}: primary disk")
    ax.set_aspect("equal")
plt.show()

## 4. What's gone
The old forms now fail with a message that says what's wrong:

In [ ]:
attempts = {
    "sim.simulate_particles(params, config, seed=1)": lambda: sim.simulate_particles(params, config, seed=1),
    "simr.run(..., {'run': {'seed': 3}})": lambda: simr.run(params, {"run": {"seed": 3}}),
    "SimulatorConfig(seed=None)": lambda: sim.SimulatorConfig(seed=None),
    "SimulatorConfig(seed=-1)": lambda: sim.SimulatorConfig(seed=-1),
}
for label, call in attempts.items():
    try:
        call()
    except Exception as error:
        print(f"{label:50s} -> {type(error).__name__}: {error}")